In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_pinball_loss
import os
import json
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from scipy.stats import uniform, randint
from sklearn.metrics import mean_squared_error
import cloudpickle
import io
import pickle

import warnings
warnings.filterwarnings("ignore")

# Model configurations for different types
MODEL_CONFIGS = {
    'xgboost': {
        'model_class': xgb.XGBRegressor,
        'param_dist': {
            'model__n_estimators': randint(50, 300),
            'model__learning_rate': uniform(0.01, 0.09),
            'model__max_depth': randint(3, 8),
            'model__subsample': uniform(0.6, 0.4),
            'model__colsample_bytree': uniform(0.6, 0.4),
            'model__min_child_weight': randint(1, 6),
            'model__gamma': uniform(0, 0.2),
            'model__reg_alpha': uniform(0, 0.5),
            'model__reg_lambda': uniform(1, 1.5),
        }
    },
    'lightgbm': {
        'model_class': lgb.LGBMRegressor,
        'param_dist': {
            'model__n_estimators': randint(50, 300),
            'model__learning_rate': uniform(0.01, 0.09),
            'model__max_depth': randint(3, 8),
            'model__subsample': uniform(0.6, 0.4),
            'model__colsample_bytree': uniform(0.6, 0.4),
            'model__min_child_weight': randint(1, 6),
            'model__reg_alpha': uniform(0, 0.5),
            'model__reg_lambda': uniform(1, 1.5),
        }
    },
    'catboost': {
        'model_class': cb.CatBoostRegressor,
        'param_dist': {
            'model__iterations': randint(50, 300),
            'model__learning_rate': uniform(0.01, 0.09),
            'model__depth': randint(3, 8),
            'model__subsample': uniform(0.6, 0.4),
            'model__colsample_bylevel': uniform(0.6, 0.4),
            'model__reg_lambda': uniform(1, 1.5),
        }
    },
    'random_forest': {
        'model_class': RandomForestRegressor,
        'param_dist': {
            'model__n_estimators': randint(50, 300),
            'model__max_depth': randint(3, 15),
            'model__min_samples_split': randint(2, 20),
            'model__min_samples_leaf': randint(1, 10),
            'model__max_features': ['sqrt', 'log2', None],
        }
    }
}

def calculate_current_month(row):
    """Calculate current month based on cohort and elapsed months"""
    cohort_start = row['cohort']
    elapsed_months = row['elapsed_months']
    return (cohort_start + pd.DateOffset(months=int(elapsed_months))).month

def quantile_loss(alpha):
    """Quantile loss function for XGBoost"""
    def custom_loss(y_true, y_pred):
        errors = y_pred - y_true
        grad = np.where(errors < 0, -alpha, 1 - alpha)
        hess = np.ones_like(y_true)
        return grad, hess
    return custom_loss

def create_model_pipeline(model_type, X_train, quantile_alpha=None):
    """Create model pipeline based on model type"""
    numerical_columns = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[('num', StandardScaler(), numerical_columns)],
        remainder='passthrough'
    )
    
    # Create model based on type
    if model_type == 'xgboost':
        if quantile_alpha is not None:
            model = xgb.XGBRegressor(objective=quantile_loss(quantile_alpha), enable_categorical=True)
        else:
            model = xgb.XGBRegressor(enable_categorical=True)
    elif model_type == 'lightgbm':
        if quantile_alpha is not None:
            model = lgb.LGBMRegressor(objective='quantile', alpha=quantile_alpha)
        else:
            model = lgb.LGBMRegressor()
    elif model_type == 'catboost':
        if quantile_alpha is not None:
            model = cb.CatBoostRegressor(loss_function=f'Quantile:alpha={quantile_alpha}', verbose=False)
        else:
            model = cb.CatBoostRegressor(verbose=False)
    elif model_type == 'random_forest':
        model = RandomForestRegressor(random_state=42)
    else:
        raise ValueError(f"Unsupported model type: {model_type}")
    
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    return pipeline

def model_train_save_pipeline(game, model_type, alpha, X_train, y_train):
    """Train and save model pipeline"""
    pipeline = create_model_pipeline(model_type, X_train, alpha)
    
    # Get parameter distribution for the model type
    if model_type in MODEL_CONFIGS:
        param_dist = MODEL_CONFIGS[model_type]['param_dist'].copy()
    else:
        param_dist = {}
    
    if param_dist:
        random_search = RandomizedSearchCV(
            pipeline, 
            param_distributions=param_dist,
            n_iter=20, 
            scoring=make_scorer(mean_pinball_loss, alpha=alpha, greater_is_better=False),
            cv=TimeSeriesSplit(n_splits=3), 
            n_jobs=-1, 
            random_state=42
        )
        random_search.fit(X_train, y_train)
        best_model = random_search.best_estimator_
    else:
        # Fallback to simple training if no hyperparameters defined
        best_model = pipeline
        best_model.fit(X_train, y_train)
    
    # Save model
    filename = f'{game}_{model_type}_{alpha}.pkl'
    with open(filename, 'wb') as f:
        cloudpickle.dump(best_model, f)
    
    # Store feature names
    best_model.__dict__["feature_names_in_"] = X_train.columns
    
    # Calculate training RMSE
    y_train_pred = best_model.predict(X_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    
    return {"model": best_model, "rmse_train": rmse_train}

def load_best_model(game_id, model_type):
    """Load the best model for a game from file"""
    try:
        filename = f'{game_id}_{model_type}_0.5.pkl'
        with open(filename, 'rb') as f:
            model = cloudpickle.load(f)
        return model
    except FileNotFoundError:
        print(f"Warning: Model file {filename} not found. Using default training.")
        return None

def prepare_training_data(model_type, df, dft, selected_features):
    """Prepare training data based on model type and selected features - matches utils.py"""
    if model_type == "M1":
        train_features = selected_features.copy()
        if 'elapsed_months' not in train_features:
            train_features.append('elapsed_months')
        
        X = df[train_features].drop(df.index[-1])
        y = df['transaction_period_ltv_cac'].drop(df.index[-1])
    else:  # M2
        additional_features = ['ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3', 'elapsed_months']
        train_features = [col for col in selected_features + additional_features if col in dft.columns]
        X = dft[train_features]
        y = dft['transaction_period_ltv_cac']
    
    return X, y

def augment_data(X, y, augmentation_factor, variation_pct):
    """Augment training data with variations"""
    n_aug = int((augmentation_factor - 1)*len(X))
    if n_aug <= 0:
        return X, y
    
    numeric_cols = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col]) and col not in ['cohort_month','current_month']]
    X_aug_list, y_aug_list = [], []
    
    for _ in range(n_aug):
        idx = np.random.randint(0, len(X))
        x_row = X.iloc[idx].copy()
        
        for col in numeric_cols:
            sign = np.random.choice([-1,1])
            x_row[col] = x_row[col] * (1 + sign*(variation_pct/100.0))
        
        X_aug_list.append(x_row)
        y_val = y.iloc[idx]
        sign = np.random.choice([-1,1])
        y_aug_list.append(y_val * (1 + sign*(variation_pct/100.0)))
    
    X_aug = pd.DataFrame(X_aug_list)
    y_aug = pd.Series(y_aug_list, index=X_aug.index)
    X_final = pd.concat([X, X_aug], ignore_index=True)
    y_final = pd.concat([y, y_aug], ignore_index=True)
    
    return X_final, y_final

def preprocess_m2(game):
    """Enhanced preprocess data for M2 model - COMPLETE match with preprocessing.py"""
    df = pd.read_csv(f'Latest Data/Segregated/{game}_Dataset_12_2024.csv')
    df['elapsed_months'] = df['elapsed_months'].fillna(1)
    df = df[(df['total_cac'] != 0) & (df['total_cac'].notna())]
    df['elapsed_months'] -= 1
    df = df[df['elapsed_months'] >= 0]
    
    # Handle total_cac and client_actual_spend
    df['total_cac'] = df.apply(
        lambda row: row['client_actual_spend'] 
        if (pd.isnull(row['total_cac']) or row['total_cac'] == 0) and not pd.isnull(row['client_actual_spend'])
        else row['total_cac'],
        axis=1
    )
    df['client_actual_spend'] = df.apply(
        lambda row: row['total_cac']
        if (pd.isnull(row['client_actual_spend']) or row['client_actual_spend'] == 0) and not pd.isnull(row['total_cac'])
        else row['client_actual_spend'],
        axis=1
    )
    
    # Filter out old cohorts
    df = df[~df['cohort'].str.endswith(('2021', '2022','2020','2019','2018'))]
    df = df[~df['cohort'].str.startswith(('2021','2022','2020','2019','2018'))]
    
    # Convert cohort to datetime
    df['cohort'] = pd.to_datetime(df['cohort'], format='%d/%m/%Y')
    df['cohort_month'] = df['cohort'].dt.month
    df['current_month'] = df.apply(calculate_current_month, axis=1)

    # Determine current month start date as a string (with formatting adjustments)
    current_month_start = pd.Timestamp.today().normalize().replace(day=1)  # first day of current month as a Timestamp (fix C1)
    
    # Convert transaction_period to datetime
    df['transaction_period'] = pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce')
    
    # Filter data before current month
    filtered_df = df[
        pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce') < current_month_start
    ].copy()

    # Identify exclude columns (EXACTLY as in preprocessing.py)
    exclude_columns = [
        'transaction_period_ltv_cac',  # target variable
        'game_id',  # identifier
        'cohort',  # used for grouping
        'transaction_period',  # used for filtering
        'elapsed_months',  # added separately
        'client_actual_spend'  # used as total_cac
    ]
    additional_excludes = ['interest', 'adjustment', 'adjustment_interest', 'client_id', 'elapsed_months.1']
    for exc in additional_excludes:
        if exc in df.columns:
            exclude_columns.append(exc)

    # Exclude date columns
    for col in df.columns:
        if 'date' in col.lower():
            exclude_columns.append(col)

    # Base features (EXACTLY as in preprocessing.py)
    mc_cols = [col for col in df.columns if col.startswith('MC_')]
    day_cols = [col for col in df.columns if col.startswith('day_')]
    base_features = mc_cols + day_cols + ['total_cac', 'm0_ltv_cac', 'cohort_month', 'current_month']

    # Remaining columns
    remaining_cols = [col for col in df.columns if col not in exclude_columns and col not in base_features]

    # 🎯 CREATE M1 FEATURES (Critical for model performance!)
    m1_df = df[df['elapsed_months'] == 1][['cohort'] + remaining_cols].copy()
    if not m1_df.empty:
        m1_df = m1_df.rename(columns={col: f'm1_feature_{col}' for col in remaining_cols})
        m1_df.set_index('cohort', inplace=True)
        for new_col in m1_df.columns:
            df[new_col] = df['cohort'].map(m1_df[new_col])
        df[m1_df.columns] = df[m1_df.columns].fillna(0)

    # Fill missing numeric values
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # 📊 FEATURE COLUMNS (Complete feature set with M1 features)
    feature_columns = base_features + [f'm1_feature_{col}' for col in remaining_cols if f'm1_feature_{col}' in df.columns]

    # Create lag features
    for lag in range(1, 4):
        df[f'ltv_lag_{lag}'] = df.groupby('cohort')['transaction_period_ltv_cac'].shift(lag)
    
    # Return filtered training data (EXACTLY as in preprocessing.py)
    dft = df[
        pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce') < current_month_start
    ].copy()
    
    return dft, feature_columns, df

def forecast_cumulative_ltv_cac_variations_m2_enhanced(game_id, best_model_expected, best_model_bad, best_model_good, df, forecast_months=12, epsilon=0.001):
    """Enhanced forecasting function supporting multiple model types"""
    game_data = df[df['game_id'] == game_id].copy()
    if game_id == "G090":
        game_data = game_data[game_data['cohort'] >= pd.Timestamp("2024-12-01")]
    
    forecasted_results = []
    sp_ltv_cac = df[df['transaction_period_ltv_cac'] > 0]['transaction_period_ltv_cac'].min()
    
    for cohort in game_data['cohort'].unique():
        cohort_data = game_data[game_data['cohort'] == cohort].copy()
        m0_ltv_cac = cohort_data['m0_ltv_cac'].iloc[0]
        total_cac = cohort_data['client_actual_spend'].iloc[0]
        last_elapsed_month = int(cohort_data['elapsed_months'].max())
        
        # Add row for elapsed_months=0 with actual data if available
        elapsed_0_data = cohort_data.loc[cohort_data['elapsed_months'] == 0]
        if not elapsed_0_data.empty:
            transactional_actual_0 = elapsed_0_data['transaction_period_ltv_cac'].values[0]
            cumulative_actual_0 = m0_ltv_cac
            absolute_revenue_actual_0 = total_cac * transactional_actual_0
        else:
            transactional_actual_0 = 0
            cumulative_actual_0 = 0
            absolute_revenue_actual_0 = 0
        
        row_result_0 = {
            'game_id': game_id,
            'cohort': cohort,
            'elapsed_months': 0,
            'current_month': cohort.strftime('%Y-%m'),
            'transactional_actual': transactional_actual_0,
            'cumulative_actual': cumulative_actual_0,
            'absolute_revenue_actual': absolute_revenue_actual_0,
            'total_cac': total_cac,
            'transactional_bad': 0,
            'cumulative_bad': cumulative_actual_0,
            'absolute_revenue_bad': absolute_revenue_actual_0,
            'transactional_expected': 0,
            'cumulative_expected': cumulative_actual_0,
            'absolute_revenue_expected': absolute_revenue_actual_0,
            'transactional_good': 0,
            'cumulative_good': cumulative_actual_0,
            'absolute_revenue_good': absolute_revenue_actual_0,
        }
        forecasted_results.append(row_result_0)
        
        # Initialize cumulative values
        cumulative_actual = cumulative_actual_0
        actual_cumulative = []
        for month in range(1, last_elapsed_month + 1):
            historical_data = cohort_data.loc[cohort_data['elapsed_months'] == month, 'transaction_period_ltv_cac']
            transactional_actual = historical_data.values[0] if not historical_data.empty else 0
            cumulative_actual += transactional_actual
            actual_cumulative.append({
                'month': month,
                'transactional_actual': transactional_actual,
                'cumulative_actual': cumulative_actual,
                'absolute_revenue_actual': total_cac * transactional_actual,
                'total_cac': total_cac
            })
        
        # Forecasting logic for elapsed_months >= 1
        lag_1 = np.nan; lag_2 = np.nan; lag_3 = np.nan
        lag_1b = np.nan; lag_1g = np.nan; lag_2b = np.nan; lag_2g = np.nan; lag_3b = np.nan; lag_3g = np.nan
        cumulative_values = {key: cumulative_actual_0 for key in ['cumulative_bad', 'cumulative_expected', 'cumulative_good']}
        
        # Get feature names - handle different model types
        feature_names = getattr(best_model_expected, 'feature_names_in_', None)
        if feature_names is None and hasattr(best_model_expected, '__dict__'):
            feature_names = best_model_expected.__dict__.get("feature_names_in_", None)
        
        if feature_names is not None:
            static_features = cohort_data.loc[cohort_data['elapsed_months'] == last_elapsed_month, feature_names].iloc[0].copy()
            for col in ['elapsed_months', 'current_month', 'ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3']:
                if col in static_features:
                    static_features = static_features.drop(col)
        else:
            print(f"Warning: No feature names found for {game_id}, using default features")
            static_features = cohort_data.loc[cohort_data['elapsed_months'] == last_elapsed_month].iloc[0].copy()
        
        for i in range(1, last_elapsed_month + forecast_months + 1):
            elapsed_months = i
            features = static_features.copy()
            features['elapsed_months'] = elapsed_months
            
            if feature_names is not None and 'current_month' in feature_names:
                cohort_start_date = pd.to_datetime(cohort_data['cohort'].iloc[0])
                current_date = cohort_start_date + pd.DateOffset(months=elapsed_months)
                features['current_month'] = current_date.month
            else:
                current_date = pd.to_datetime(cohort_data['cohort'].iloc[0]) + pd.DateOffset(months=elapsed_months)
            
            if i <= last_elapsed_month:
                historical_data = cohort_data.loc[cohort_data['elapsed_months'] == i, 'transaction_period_ltv_cac']
                if not historical_data.empty:
                    lag_value = historical_data.values[0] if historical_data.values[0] != 0 else epsilon
                else:
                    lag_value = epsilon
                lag_1 = lag_value
                lag_1b = lag_value
                lag_1g = lag_value
                predicted_val_bad = lag_value
                predicted_val_expected = lag_value
                predicted_val_good = lag_value
            else:
                features1 = features.copy()
                features2 = features.copy()
                features['ltv_lag_1'] = lag_1
                features1['ltv_lag_1'] = lag_1b
                features2['ltv_lag_1'] = lag_1g
                features['ltv_lag_2'] = lag_2
                features1['ltv_lag_2'] = lag_2b
                features2['ltv_lag_2'] = lag_2g
                features['ltv_lag_3'] = lag_3
                features1['ltv_lag_3'] = lag_3b
                features2['ltv_lag_3'] = lag_3g
                
                features_df = pd.DataFrame([features])
                features_df1 = pd.DataFrame([features1])
                features_df2 = pd.DataFrame([features2])
                
                predicted_val_bad = max(0, best_model_bad.predict(features_df1)[0])
                predicted_val_expected = max(0, best_model_expected.predict(features_df)[0])
                predicted_val_good = max(0, best_model_good.predict(features_df2)[0])
                
                x = [predicted_val_bad, predicted_val_expected, predicted_val_good]
                x.sort()
                predicted_val_bad, predicted_val_expected, predicted_val_good = x
                
                if predicted_val_good <= 0:
                    predicted_val_good = sp_ltv_cac
                if predicted_val_expected <= 0:
                    predicted_val_expected = predicted_val_good / 2
                if predicted_val_bad <= 0:
                    predicted_val_bad = predicted_val_expected / 2
                
                lag_1b = predicted_val_bad
                lag_1 = predicted_val_expected
                lag_1g = predicted_val_good
            
            lag_3 = lag_2
            lag_2 = lag_1
            lag_3b = lag_2b
            lag_2b = lag_1b
            lag_3g = lag_2g
            lag_2g = lag_1g
            
            row_result = {
                'game_id': game_id,
                'cohort': cohort,
                'elapsed_months': elapsed_months,
                'current_month': current_date.strftime('%Y-%m'),
            }
            
            actual_data = next((ac for ac in actual_cumulative if ac['month'] == elapsed_months), None)
            row_result['transactional_actual'] = actual_data['transactional_actual'] if actual_data else None
            row_result['cumulative_actual'] = actual_data['cumulative_actual'] if actual_data else None
            row_result['absolute_revenue_actual'] = actual_data['absolute_revenue_actual'] if actual_data else None
            row_result['total_cac'] = actual_data['total_cac'] if actual_data else total_cac
            
            for model_type, predicted_val in zip(['bad', 'expected', 'good'], [predicted_val_bad, predicted_val_expected, predicted_val_good]):
                cumulative_key = f"cumulative_{model_type}"
                cumulative_values[cumulative_key] += predicted_val
                row_result[f"transactional_{model_type}"] = predicted_val
                row_result[cumulative_key] = cumulative_values[cumulative_key]
                row_result[f"absolute_revenue_{model_type}"] = total_cac * predicted_val
            
            forecasted_results.append(row_result)
    
    forecast_df = pd.DataFrame(forecasted_results)
    return forecast_df

# --- Main Execution for Enhanced M2 ---
if __name__ == "__main__":
    config_path = "Latest Data/Segregated/config1.json"
    
    try:
        with open(config_path, "r") as f:
            global_config = json.load(f)
        print(f"✅ Loaded config from {config_path}")
    except Exception as e:
        print(f"❌ No config found or error reading config1.json: {e}")
        global_config = {}
    
    failed_games = []
    changed_features = {}
    
    # Process each M2 configuration
    for key, cfg in global_config.items():
        if key.startswith("M2_") and cfg.get("flag", 0) == 1:
            game_id = cfg["game_id"]
            selected_features = cfg["selected_features"]
            best_model_type = cfg.get("best_model", "xgboost")  # Default to xgboost if not specified
            
            print(f"\n🎮 Processing M2 configuration for {game_id} with {best_model_type} model...")
            
            try:
                # Preprocess data using COMPLETE enhanced preprocessing (matches app.py)
                dft, feature_columns, df_orig = preprocess_m2(game_id)
                
                print(f"📊 Available feature types:")
                mc_features = [f for f in feature_columns if f.startswith('MC_')]
                day_features = [f for f in feature_columns if f.startswith('day_')]
                m1_features = [f for f in feature_columns if f.startswith('m1_feature_')]
                base_features = [f for f in feature_columns if f in ['total_cac', 'm0_ltv_cac', 'cohort_month', 'current_month']]
                
                print(f"   🎯 Base features: {len(base_features)} ({base_features})")
                print(f"   📈 MC features: {len(mc_features)}")
                print(f"   📊 Day features: {len(day_features)}")
                print(f"   🔥 M1 features: {len(m1_features)}")
                
                # Check for missing features
                missing_feats = [feat for feat in selected_features if feat not in dft.columns]
                if missing_feats:
                    print(f"⚠️  Warning: For game {game_id}, the following features are missing and will be ignored: {missing_feats}")
                    changed_features[game_id] = missing_feats
                    selected_features = [feat for feat in selected_features if feat in dft.columns]
                
                # Prepare training data using the same logic as app.py
                X, y = prepare_training_data("M2", df_orig, dft, selected_features)
                
                # Data augmentation (if configured)
                use_augmentation = cfg.get("use_augmentation", False)
                augmentation_factor = cfg.get("augmentation_factor", 1.0)
                augmentation_pct = cfg.get("augmentation_pct", 0)
                
                X_train, y_train = X, y
                if use_augmentation and augmentation_factor > 1:
                    print(f"🔄 Applying data augmentation (factor: {augmentation_factor}, variation: {augmentation_pct}%)")
                    X_train, y_train = augment_data(X_train, y_train, augmentation_factor, augmentation_pct)
                
                # Train models for quantile regression (bad, expected, good scenarios)
                alphas = [0.25, 0.5, 0.75]
                models_info = {}
                
                for alpha in alphas:
                    print(f"🏋️  Training {best_model_type} model with alpha={alpha} for game {game_id}...")
                    result = model_train_save_pipeline(game_id, best_model_type, alpha, X_train, y_train)
                    models_info[alpha] = result
                
                # Load the trained models
                best_model_expected = models_info[0.5]["model"]
                best_model_bad = models_info[0.75]["model"]
                best_model_good = models_info[0.25]["model"]
                
                # Generate forecasts
                print(f"📊 Generating forecasts for {game_id}...")
                forecast_df = forecast_cumulative_ltv_cac_variations_m2_enhanced(
                    game_id, best_model_expected, best_model_bad, best_model_good, 
                    df_orig, forecast_months=12
                )
                
                # Save predictions
                output_csv = f"{game_id}_predictions.csv"
                forecast_df.to_csv(output_csv, index=False)
                print(f"✅ Predictions saved to {output_csv}")
                
            except Exception as e:
                print(f"❌ Error processing game {game_id}: {e}")
                failed_games.append(game_id)
                continue
    
    # Summary
    print(f"\n🎉 Processing completed!")
    
    if failed_games:
        print(f"\n⚠️  The following {len(failed_games)} games failed during training:")
        for game in failed_games:
            print(f"   • {game}")
    
    if changed_features:
        print(f"\n🔧 The following {len(changed_features)} games had missing features (these features were ignored):")
        for game, features in changed_features.items():
            print(f"   • {game}: Missing features {features}")
    
    successful_games = [key.replace("M2_", "") for key in global_config.keys() 
                       if key.startswith("M2_") and global_config[key].get("flag", 0) == 1 
                       and key.replace("M2_", "") not in failed_games]
    
    print(f"\n✅ Successfully processed {len(successful_games)} games")
    if successful_games:
        print("   Games processed:")
        for game in successful_games:
            print(f"   • {game}") 